# Your first neural network

**Foundations of AI, Saturday 26 September 2026. Homework.**

Today you watched a machine guess, check and correct until it could price a
house. Now you build one yourself, and by the end of this page it will look at
a flower it has never seen and say what it is.

About an hour, reading included. No software, no installation, no experience.
Almost all of the code is written for you.

Everywhere you see 🎯 it is your turn. There are five of them.
Everything else runs by itself.

## Before you start: how this page works

This is a **Colab notebook**: a web page made of two kinds of block. Text, like
this one, and small blocks of **code** that you run.

**To run a block of code**, move your mouse over it. A round play button appears
on its left. Click it. You can also press **Shift and Enter** together.

**Run them in order, from the top.** Each block uses what the ones above it have
already done, so skipping one produces an error further down.

**While a block runs** it shows a spinning circle. When it has finished, the
circle turns into a small number. Most blocks here finish instantly, and the
slowest takes about two seconds.

**The first time you run anything**, Google warns you that this notebook was not
written by Google. Click **Run anyway**. You also have to be signed in to a
Google account.

**To keep your changes**, use *File, Save a copy in Drive* before you start.
Otherwise everything disappears when you close the tab, which is fine for this.

**A red box is normal.** If you run a block before filling in its blank, or
fill it in wrongly, the block turns red. That is not a disaster and nothing is
broken. Read the **last line** of the red box, which is the one written in
plain words, fix the blank and run the block again. You can do that as often as
you like.

**If you get really stuck**, use *Runtime, Restart session* and run the blocks
again from the top. Nothing here can break anything, on your computer or
anywhere else.

A block that appears as a grey bar with a title is one you never need to read.
It sets something up. Run it and move on. The first one is below.

In [ ]:
# @title Run this cell first (it loads the flowers and the checkers) { display-mode: "form" }

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

torch.manual_seed(0)

iris = load_iris()
SPECIES = list(iris.target_names)

X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

MEASUREMENTS = ["sepal length", "sepal width", "petal length", "petal width"]
MODEL_FOR_CHECK = [None]
OPTIMIZER_FOR_CHECK = [None]

# The completion code. One bit per bullseye: did its checker pass the FIRST
# time it ran. The participant may then try as often as they like.

_ALPHABET = "ABCDEFGHJKMNPQRSTUVWXYZ"
_SALT = "FAI-HS26-NB1"
_NQ = 5
_first = {}


def _fnv1a32(text):
    h = 0x811C9DC5
    for b in text.encode("ascii"):
        h ^= b
        h = (h * 0x01000193) & 0xFFFFFFFF
    return h


def _seed():
    if "seed" not in _first:
        import random
        r = random.SystemRandom()
        _first["seed"] = "".join(r.choice(_ALPHABET) for _ in range(4))
    return _first["seed"]


def completion_code():
    seed = _seed()
    bits = 0
    for i in range(_NQ):
        if _first.get(i + 1) is True:
            bits |= 1 << i
    tag = bits ^ (_fnv1a32(_SALT + "|" + seed) % (1 << _NQ))
    out = ""
    v = tag
    for _ in range(3):
        out = _ALPHABET[v % 23] + out
        v //= 23
    chk = _ALPHABET[_fnv1a32(_SALT + "|" + seed + "|" + out) % 23]
    return seed + out + chk


def _record(n, ok):
    if n not in _first:
        _first[n] = ok


def _say(ok, n, good, bad):
    _record(n, ok)
    if ok:
        print("Right. " + good)
    else:
        print("Not yet. " + bad)
    return ok


def check_1(net):
    layers = list(net)
    linear = [m for m in layers if isinstance(m, nn.Linear)]
    bends = [m for m in layers if isinstance(m, (nn.ReLU, nn.Tanh, nn.Sigmoid))]
    if not bends:
        return _say(False, 1, "",
                    "There is no bend between the two layers. Without one, two layers do "
                    "exactly what one layer does, and the network can only draw a straight "
                    "line. Write nn.ReLU() on that line.")
    if len(linear) < 2 or linear[-1].out_features != 3:
        got = linear[-1].out_features if linear else 0
        return _say(False, 1, "",
                    "The last layer gives %d score(s). There are three species, so it has "
                    "to give three, one per species." % got)
    out = net(X_train[:1])
    if tuple(out.shape) != (1, 3):
        return _say(False, 1, "",
                    "The network answers with the wrong shape: %s." % (tuple(out.shape),))
    return _say(True, 1,
                "Four measurements go in, the first layer turns them into 8 numbers, the "
                "bend throws away the negative ones, and the last layer turns those into 3 "
                "scores, one per species.", "")


def check_2(opt, net):
    given = []
    for group in opt.param_groups:
        given.extend(group["params"])
    wanted = list(net.parameters())
    if not given:
        return _say(False, 2, "",
                    "The optimizer has been given nothing to turn. It needs the list of "
                    "the network's knobs, which is model.parameters()")
    if len(given) != len(wanted) or any(a is not b for a, b in zip(given, wanted)):
        return _say(False, 2, "",
                    "Those are not this network's knobs. Pass model.parameters(), with the "
                    "brackets, so the optimizer turns the layers you just built.")
    _watch(opt)
    OPTIMIZER_FOR_CHECK[0] = opt
    n = sum(p.numel() for p in wanted)
    return _say(True, 2,
                "The optimizer now has all %d knobs of your network and may turn every one "
                "of them." % n, "")


def _watch(opt):
    # Remember the order in which zero_grad and step are called, so that a
    # training loop missing one of them cannot quietly pass. On this data a
    # loop that never clears the tilts still converges, which is exactly the
    # false pass this exists to catch.
    if getattr(opt, "_calls", None) is not None:
        return
    opt._calls = []
    real_zero = opt.zero_grad
    real_step = opt.step

    def zero_grad(*a, **k):
        opt._calls.append("clear")
        return real_zero(*a, **k)

    def step(*a, **k):
        opt._calls.append("step")
        return real_step(*a, **k)

    opt.zero_grad = zero_grad
    opt.step = step


def check_3(history, epochs):
    if len(history) == 0:
        return _say(False, 3, "",
                    "The loop body never ran, so nothing was written down. The loop has to "
                    "go over range(EPOCHS).")
    if len(history) != epochs:
        return _say(False, 3, "",
                    "The loop ran %d times and EPOCHS is %d. Use range(EPOCHS)."
                    % (len(history), epochs))
    return _say(True, 3,
                "%d passes over the flowers, with a loss written down after every one."
                % epochs, "")


def check_4(history, net):
    if len(history) < 2:
        return _say(False, 4, "", "Nothing was trained, so there is nothing to check yet.")
    calls = getattr(OPTIMIZER_FOR_CHECK[0], "_calls", None) or []
    tail = calls[-2 * len(history):]
    if tail.count("clear") == 0:
        return _say(False, 4, "",
                    "The old tilts are never cleared. PyTorch adds every new tilt on top of "
                    "the ones already there, so after two hundred passes it is walking on a "
                    "sum of everything it ever measured. Line 4a is optimizer.zero_grad()")
    if tail.count("step") == 0:
        return _say(False, 4, "",
                    "Nothing ever takes a step, so no knob was turned. Line 4c is "
                    "optimizer.step()")
    if tail and tail[0] != "clear":
        return _say(False, 4, "",
                    "The step is being taken before the tilts are cleared. Clearing comes "
                    "first, then working out the new tilts, then the step: you cannot walk "
                    "downhill before you have looked at the ground.")
    with torch.no_grad():
        right = (net(X_train).argmax(1) == y_train).float().mean().item()
    if history[-1] > 0.35 or right < 0.85:
        return _say(False, 4, "",
                    "The loss started at %.2f and ended at %.2f, and the network gets only "
                    "%.0f%% of the training flowers right. The three lines are in the wrong "
                    "order. Clearing the old tilts has to happen before the new ones are "
                    "worked out, and the step has to come last: you cannot walk downhill "
                    "before you have looked at the ground."
                    % (history[0], history[-1], 100 * right))
    return _say(True, 4,
                "The loss fell from %.2f to %.2f. That is guess, check, correct, two "
                "hundred times over, and nobody told the network what an iris looks like."
                % (history[0], history[-1]), "")


def check_5(pred):
    with torch.no_grad():
        want = MODEL_FOR_CHECK[0](X_test).argmax(1)
    if not torch.is_tensor(pred):
        return _say(False, 5, "", "That is not a list of predictions.")
    if pred.shape != want.shape:
        return _say(False, 5, "",
                    "Your predictions have shape %s and there are %d test flowers. "
                    "argmax(1) looks along each flower's own three scores; argmax(0) looks "
                    "down the wrong way." % (tuple(pred.shape), want.shape[0]))
    if not torch.equal(pred, want):
        return _say(False, 5, "", "Those are not the highest scoring species.")
    right = (pred == y_test).float().mean().item()
    return _say(True, 5,
                "%.0f%% of thirty flowers the network had never seen. That is the only "
                "number worth quoting." % (100 * right), "")


def build_network(measurements, hidden, bend, scores):
    if bend is Ellipsis:
        raise ValueError("Bullseye 1a is still blank. The bend goes there: write nn.ReLU()")
    if not isinstance(bend, nn.Module):
        raise ValueError("That is not a bend. Write nn.ReLU(), with the brackets.")
    if scores is Ellipsis:
        raise ValueError("Bullseye 1b is still blank. How many species are there?")
    if not isinstance(scores, int) or isinstance(scores, bool) or scores < 1:
        raise ValueError("The number of scores has to be a whole number. Count the species.")
    return nn.Sequential(nn.Linear(measurements, hidden), bend, nn.Linear(hidden, scores))


def show_flowers():
    print("150 flowers. Four measurements each, in centimetres.")
    print()
    print("   %-13s %-12s %-13s %-12s  species" % tuple(MEASUREMENTS))
    for i in [0, 1, 60, 61, 130, 131]:
        row = iris.data[i]
        print("   %-13.1f %-12.1f %-13.1f %-12.1f  %s"
              % (row[0], row[1], row[2], row[3], SPECIES[iris.target[i]]))
    print()
    print("The three species are: %s" % ", ".join(SPECIES))
    print("%d flowers to learn from, %d kept back to test on."
          % (X_train.shape[0], X_test.shape[0]))


def plot_loss(history):
    plt.figure(figsize=(7, 3.2))
    plt.plot(history, color="#0072B2")
    plt.xlabel("pass over the flowers")
    plt.ylabel("how wrong the network is")
    plt.title("The loss, as training goes on")
    plt.grid(True, alpha=0.3)
    plt.show()


def one_flower(net):
    flower = X_test[0].unsqueeze(0)
    with torch.no_grad():
        chances = torch.softmax(net(flower), dim=1)[0]
    print("One flower the network never saw:")
    for name, value in zip(MEASUREMENTS, X_test[0].tolist()):
        print("   %-14s %.1f cm" % (name, value))
    print()
    print("What the network says:")
    for i, name in enumerate(SPECIES):
        bar = "#" * int(round(30 * chances[i].item()))
        print("   %-12s %5.1f%%  %s" % (name, 100 * chances[i].item(), bar))
    print()
    print("It really is: %s" % SPECIES[y_test[0].item()])


def train_again(rate, passes):
    torch.manual_seed(0)
    net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 3))
    fn = nn.CrossEntropyLoss()
    opt = torch.optim.SGD(net.parameters(), lr=rate)
    history = []
    for _ in range(passes):
        loss = fn(net(X_train), y_train)
        opt.zero_grad()
        loss.backward()
        opt.step()
        history.append(loss.item())
    with torch.no_grad():
        acc = (net(X_test).argmax(1) == y_test).float().mean().item()
    print("step size %-10s   loss at the end %8.3f   test flowers right %5.1f%%"
          % (rate, history[-1], 100 * acc))
    plot_loss(history)


print("Ready. 150 flowers loaded, nothing installed, nothing downloaded.")


## 1. The flowers

Nothing to fill in. Run the block and look at what we have.

In [ ]:
show_flowers()

Each flower is **four numbers**, and belongs to **one of three species**.
The network's job is to turn those four numbers into the right species.

Notice the last line. Thirty flowers are put in a drawer, and the network is
never shown them while it learns. Those thirty are the only honest test of
whether it learned anything, which is the point that kept coming back today.

## 2. The network

This morning you saw what a neuron is: a **weighted sum, and then a bend**.
Stack those into layers and you have a neural network.

Ours is about as small as a network gets:

```
   4 measurements  ->  8 hidden numbers  ->  bend  ->  3 scores
```

The three scores at the end are one per species, and the biggest score wins.

🎯 **1. Two blanks.**

**a)** The bend. Without it, two layers do exactly what one layer does and the
network can only ever draw a straight line. In PyTorch the bend is `nn.ReLU()`,
and it does one thing: every negative number becomes zero.

**b)** How many scores the last layer gives. Count the species.

In [ ]:
model = build_network(
    measurements = 4,   # four numbers per flower go in
    hidden = 8,         # eight numbers in the middle
    bend = ...,         # 🎯 1a) the bend goes here
    scores = ...,       # 🎯 1b) one score per species
)

print(model)
MODEL_FOR_CHECK[0] = model
check_1(model)

## 3. What to measure, and what to turn

Two things before training.

The **loss** is the single number saying how wrong the network is right now. For
a choice between species the usual one is called cross entropy, and it is
written for you below.

The **optimizer** is what actually turns the knobs. It needs two things: which
knobs it may turn, and how big a step to take.

🎯 **2.** Tell the optimizer which knobs it may turn. Any
network in PyTorch will list its own if you ask it: `model.parameters()`

In [ ]:
loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(
    ...,        # 🎯 2) which knobs the optimizer may turn
    lr=0.05,    # the step size, from this morning
)

check_2(optimizer, model)

`lr` is the **step size**: the very number that made the machine bounce for
ever this morning when it was too big. Section 7 lets you play with it.

## 4. Training

This is the loop from the whole day. Once per pass over the flowers:

1. **guess** what the species are,
2. **check** how wrong that was, which is the loss,
3. **correct** every knob a little, against the tilt.

🎯 **3.** The loop itself. The network goes over all the
flowers `EPOCHS` times. Use `range(EPOCHS)`.

🎯 **4.** The three lines that make PyTorch learn. Use each of
these **once**, and work out which goes where:

```
optimizer.zero_grad()      loss.backward()      optimizer.step()
```

- **a)** clear the tilts left over from the last pass, because PyTorch adds new
  ones on top of whatever is already there
- **b)** work out, for every knob, which way makes this loss smaller
- **c)** nudge every knob one step in that direction

Think about which has to happen before which. Getting it wrong breaks nothing,
and the checker will tell you what went where.

In [ ]:
EPOCHS = 200
loss_history = []

for epoch in ...:    # 🎯 3) go over the flowers EPOCHS times

    scores = model(X_train)             # the guess
    loss = loss_fn(scores, y_train)     # how wrong the guess was

    ...    # 🎯 4a) clear the old tilts
    ...    # 🎯 4b) work out the new tilts
    ...    # 🎯 4c) take one step

    loss_history.append(loss.item())

check_3(loss_history, EPOCHS)
check_4(loss_history, model)

### The learning curve

Nothing to fill in. This is the picture of the loss falling, the one behind
every training run you will ever be shown.

In [ ]:
plot_loss(loss_history)

## 5. The two numbers

The network now gets almost every **training** flower right. On its own that
number is worth nothing, because it saw those flowers while it learned.

The thirty in the drawer are the honest test.

🎯 **5.** For each test flower the network gives three scores.
Pick the species with the highest score. `argmax(1)` does exactly that, and the
`1` means look along each flower's own three scores.

In [ ]:
with torch.no_grad():                 # only looking, not learning
    test_scores = model(X_test)

predicted = ...    # 🎯 5) the species with the highest score

check_5(predicted)

That percentage, on flowers the network never saw, is the only number you
would quote to anybody. It is the answer to *"measured on what, and had the
model seen it?"*

## 6. One flower, up close

Nothing to fill in. This is what the network really produces: a confidence in
each of the three species.

In [ ]:
one_flower(model)

## 7. Break it on purpose

No blanks here, and this is the most useful part of the page.

The step size is the one knob that has to be right. Change the number below and
run the block. It trains a fresh network from scratch and shows what happened.

Try **0.05** first, the one you used. Then **0.0001**. Then **5**.

- too small and it crawls: the loss barely moves in two hundred passes
- about right and it settles
- too big and it never settles at all

This is the grasshopper, and it is why a retrain slips by three weeks.

In [ ]:
STEP_SIZE = 0.05  # @param {type:"number"}
PASSES = 200  # @param {type:"integer"}

train_again(STEP_SIZE, PASSES)

## 8. Done

You built a neural network, trained it, and tested it on flowers it had never
seen. Every idea in it was on the slides today:

| What you wrote | Where you saw it |
|---|---|
| the bend | a neuron is a weighted sum, then a bend |
| `model.parameters()` | the knobs that training turns |
| `lr`, the step size | the step that has to be right |
| the loop, guess, check, correct | the whole of block 1 |
| the thirty flowers in the drawer | measured on what, and had it seen it |

Run the last block for your code, and send it back the way you were asked to. It
carries how you did and nothing else: no name, nothing about your computer.

In [ ]:
print(completion_code())

If you want to keep going, change the `8` in section 2 to `2`, then run
everything from there again. Two hidden numbers cannot separate three species,
and you can watch it fail.

Thank you for coming on a Saturday.